<a href="https://colab.research.google.com/github/guupiii/ESAA/blob/main/ESAA_OB_week4_1_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://dacon.io/competitions/official/236664/codeshare/13926

데이콘 x BDA 제 2회 학습자 수료 예측 AI 경진대회

* 코드 흐름 요약

1. 데이터 로드 및 확인
학습/평가 데이터 간 분포 및 구조 차이 존재 확인

2. EDA 수행
학습/평가 데이터 간 분포 차이 분석
범주형 변수의 범주 수 및 영향력 확인

3. 데이터 전처리
복잡한 효과가 낮은 컬럼 제거
일부 희소 범주는 ‘기타’로 통합
모델별 입력 데이터 분리

4. 모델 학습
Logistic Regression (LR)
CatBoost
XGBoost, LightGBM, RandomForest, SVM 등 비교

> 다양한 모델 비교 후 LR을 Base Model로 선정

5. 모델 평가 및 선택
F1-score 기준 평가
FN 감소가 중요하다는 점 확인
LR이 가장 안정적인 성능 및 확률 분포 제공

6. 예측 전략
기본 예측을 전부 1로 설정
예측 확률이 낮은 순으로 일부만 0으로 변경
threshold 기반보다 안정적인 성능 확보

* 새롭게 알게 된 내용 / 어려운 내용 / 배운 점

1. 새롭게 알게 된 내용
threshold 기반보다 비율 기반 예측(flip-rate)이 더 안정적일 수 있음

하나의 모델보다 여러 모델을 활용한 보정 구조가 성능 향상에 효과적임

2. 어려웠던 내용

flip-rate(비율 고정) 전략을 설계하는 기준 설정

Rescue/Demote 기준 및 적용 범위 설정

3. 배운 점

모델 성능 향상은 단순 모델 선택보다 후처리 전략이 더 중요할 수 있음

In [ ]:

def compute_lr_boundary(prob: np.ndarray, flip_rate: float):
    prob = np.asarray(prob).reshape(-1).astype(np.float64)
    n = len(prob)
    k = int(round(n * float(flip_rate)))
    k = max(0, min(k, n))
    order = np.argsort(prob, kind="mergesort")
    if n == 0:
        return 0.0, 0.0, 0.0, 0
    if k <= 0:
        last_zero = prob[order[0]]
        first_one = prob[order[0]]
    elif k >= n:
        last_zero = prob[order[-1]]
        first_one = prob[order[-1]]
    else:
        last_zero = prob[order[k - 1]]
        first_one = prob[order[k]]
    boundary = 0.5 * (float(last_zero) + float(first_one))
    return float(last_zero), float(first_one), float(boundary), int(k)

In [ ]:
dA_cat_cols = [c for c in cb_cat_cols if c in train_dA.columns]

DEPTH = 6
D_A_BASE = dict(
    loss_function="Logloss",
    eval_metric="AUC",
    iterations=None,
    learning_rate=None,
    depth=DEPTH,
    random_seed=SEED,
    thread_count=1,
    allow_writing_files=False,
)


In [ ]:
def _merge_rare_categories(train_s, test_s, min_count=RARE_MERGE_MIN_COUNT):
    tr = train_s.astype("object")
    te = test_s.astype("object")
    tr = tr.mask(tr.isna(), MISSING_TOKEN)
    te = te.mask(te.isna(), MISSING_TOKEN)

    vc = tr.value_counts(dropna=False)
    keep = {lv for lv, cnt in vc.items() if (lv != MISSING_TOKEN) and (int(cnt) >= int(min_count))}
    keep.add(MISSING_TOKEN)

    return tr.where(tr.isin(keep), OTHER_LABEL), te.where(te.isin(keep), OTHER_LABEL)